# ReavaliaÃ§Ã£o com reais NATIVAS (sem reamostragem) e controle positivo sem JPEG
**PrÃ©-requisitos em `MyDrive/tcc_politicos/`:** `real_nativo.zip` (recortes 512Ã—512 PNG dos originais do Commons), pasta `fake_geradas/` (do notebook 2).
**CondiÃ§Ãµes (todas com recorte nativo 512Ã—512, sem reamostragem):**
- **C:** sem nenhum JPEG novo (PNG nas duas classes). Ã‰ o melhor caso para os detectores (**controle positivo**).
- **A:** JPEG q90 nas duas classes.
- **D:** JPEG q90 nas reais e JPEG q95 + q90 nas falsas (equipara o nÃºmero de compressÃµes, jÃ¡ que as fotos reais jÃ¡ vÃªm de JPEG de cÃ¢mera).
Para cada condiÃ§Ã£o: auditoria de viÃ©s (atalhos triviais), AUC dos detectores com IC95%, e o subconjunto pareado por escore de propensÃ£o.
Funciona sem GPU (UnivFD em CPU; FSD numa amostra), sÃ³ mais devagar.

In [ ]:
# 1) Setup
import os, io, subprocess, zipfile, glob, numpy as np, pandas as pd
from PIL import Image
from tqdm.auto import tqdm
print(subprocess.run("nvidia-smi --query-gpu=name --format=csv", shell=True, capture_output=True, text=True).stdout or "sem GPU (CPU)")
from google.colab import drive; drive.mount('/content/drive')
ROOT = "/content/drive/MyDrive/tcc_politicos"; OUT = f"{ROOT}/nativo"; os.makedirs(OUT, exist_ok=True)
def sh(c): r = subprocess.run(c, shell=True, capture_output=True, text=True); print((r.stdout + r.stderr)[-1200:])
def unzip_fix(zpath, dest="/content"):   # zip criado no Windows usa "\" nos caminhos
    with zipfile.ZipFile(zpath) as z:
        for i in z.infolist():
            n = i.filename.replace("\\", "/")
            if n.endswith("/"): continue
            p = os.path.join(dest, n); os.makedirs(os.path.dirname(p), exist_ok=True)
            with open(p, "wb") as f: f.write(z.read(i))
if not os.path.isdir("/content/real_nativo") or not os.listdir("/content/real_nativo"): unzip_fix(f"{ROOT}/real_nativo.zip")
print(len(os.listdir("/content/real_nativo")), "recortes nativos de reais")
sh("pip -q uninstall -y torchao 2>&1 | tail -1; pip -q install ftfy regex pillow-heif click tqdm huggingface_hub 2>&1 | tail -1")
sh("cd /content && rm -rf UniversalFakeDetect Forensic-Self-Descriptions-CVPR25 && git clone -q https://github.com/Yuheng-Li/UniversalFakeDetect && git clone -q https://github.com/ductai199x/Forensic-Self-Descriptions-CVPR25.git")
sh("pip -q install --no-deps -e /content/Forensic-Self-Descriptions-CVPR25 2>&1 | tail -1")

In [ ]:
# 2) Monta as condiÃ§Ãµes C (PNG), A (JPEG q90) e D (falsas: q95 + q90). Recorte nativo 512x512 alinhado a 8 px.
def crop512(im):
    im = im.convert("RGB"); w, h = im.size
    l, t = ((w - 512) // 2 // 8) * 8, ((h - 512) // 2 // 8) * 8
    return im.crop((l, t, l + 512, t + 512))
def jpg(im, q):
    b = io.BytesIO(); im.save(b, "JPEG", quality=q); b.seek(0); return Image.open(b).convert("RGB")
gl = pd.read_csv(f"{ROOT}/geradas_log.csv") if os.path.exists(f"{ROOT}/geradas_log.csv") else pd.DataFrame(columns=["arquivo", "gerador"])
gmap = dict(zip(gl.arquivo, gl.gerador))
items = [(f, 0, "real") for f in sorted(glob.glob("/content/real_nativo/*.png"))] + [(f, 1, gmap.get(os.path.basename(f), os.path.basename(f).split("_")[0])) for f in sorted(glob.glob(f"{ROOT}/fake_geradas/*.png"))]
for c in "CAD":
    for k in ("real", "fake"): os.makedirs(f"/content/cond/{c}/{k}", exist_ok=True)
meta = []
for f, y, gen in tqdm(items):
    im = Image.open(f)
    if min(im.size) < 512: continue
    im = crop512(im); k = "fake" if y else "real"; base = os.path.splitext(os.path.basename(f))[0]
    pC = f"/content/cond/C/{k}/{base}.png"; im.save(pC)
    pA = f"/content/cond/A/{k}/{base}.jpg"; im.save(pA, "JPEG", quality=90)
    pD = f"/content/cond/D/{k}/{base}.jpg"; (jpg(im, 95) if y else im).save(pD, "JPEG", quality=90)
    meta.append(dict(base=base, label=y, gerador=gen, pC=pC, pA=pA, pD=pD))
M = pd.DataFrame(meta); M.to_csv(f"{OUT}/meta_nativo.csv", index=False)
print(len(M), "imagens"); print(M.groupby(["label", "gerador"]).size())

In [ ]:
# 3) AUDITORIA DE VIES por condicao: atalhos triviais (AUC ~0,5 = bom). Roda em CPU.
from scipy import ndimage
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
names = ["tam_kb", "laplaciano", "gradiente", "R", "G", "B", "sdR", "sdG", "sdB", "saturacao", "baixa_freq"]
def feats(p):
    a = np.asarray(Image.open(p).convert("RGB"), dtype=np.float32) / 255; g = a.mean(2)
    lap = ndimage.laplace(g); gx, gy = np.gradient(g)
    return [os.path.getsize(p) / 1024, lap.var(), np.hypot(gx, gy).mean(), *a.mean((0, 1)), *a.std((0, 1)), (a.max(2) - a.min(2)).mean(), np.abs(np.fft.fftshift(np.fft.fft2(g)))[240:272, 240:272].mean()]
y = M.label.values; FEAT = {}
def audit(Xd, yd):
    p = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)), Xd, yd, cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")[:, 1]
    return roc_auc_score(yd, p), p
res = []
for c in "CAD":
    FEAT[c] = np.array([feats(p) for p in tqdm(M[f"p{c}"], desc=f"features {c}")])
    auc, _ = audit(FEAT[c], y); ind = {n: round(max(roc_auc_score(y, FEAT[c][:, i]), 1 - roc_auc_score(y, FEAT[c][:, i])), 3) for i, n in enumerate(names)}
    res.append(dict(condicao=c, AUC_atalhos=round(auc, 3), **{k: ind[k] for k in ("laplaciano", "tam_kb", "gradiente")}))
    print(c, "AUC atalhos (todos):", round(auc, 3), "| medianas reais x falsas (laplaciano):", np.median(FEAT[c][y == 0, 1]).round(4), np.median(FEAT[c][y == 1, 1]).round(4))
A = pd.DataFrame(res); print(A.to_string(index=False)); A.to_csv(f"{OUT}/auditoria_por_condicao.csv", index=False)
print("Compare com a auditoria anterior (miniaturas do Commons): AUC 0,946. Se cair bem, as reais nativas reduziram o viÃ©s.")

In [ ]:
# 4) Detectores por condicao. Com GPU roda tudo; sem GPU: UnivFD so na condicao C e FSD numa amostra.
import sys, torch, torchvision.transforms as T
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"; print("dispositivo:", DEVICE)
RUN_FSD = True
FSD_N = None if DEVICE == "cuda" else 120
CONDS = ["C", "A", "D"] if DEVICE == "cuda" else ["C"]
sys.path.insert(0, "/content/UniversalFakeDetect"); os.chdir("/content/UniversalFakeDetect")
from models import get_model
um = get_model("CLIP:ViT-L/14"); um.fc.load_state_dict(torch.load("pretrained_weights/fc_weights.pth", map_location="cpu")); um.eval().to(DEVICE)
tf = T.Compose([T.CenterCrop(224), T.ToTensor(), T.Normalize([0.48145466, 0.4578275, 0.40821073], [0.26862954, 0.26130258, 0.27577711])])
@torch.no_grad()
def u_score(p): return um(tf(Image.open(p).convert("RGB")).unsqueeze(0).to(DEVICE)).sigmoid().item()
for c in CONDS:
    if f"u_{c}" in M and M[f"u_{c}"].notna().all(): continue
    M[f"u_{c}"] = [u_score(p) for p in tqdm(M[f"p{c}"], desc=f"UnivFD {c}")]; M.to_csv(f"{OUT}/resultados_nativo.csv", index=False)
if RUN_FSD:
    sys.path.insert(0, "/content/Forensic-Self-Descriptions-CVPR25")
    from fsd import FSDDetector
    det = FSDDetector.load(device=DEVICE); det.config["fsd"]["max_size"] = 512
    def f_score(p):
        try: z = float(det.score(p).z_score)
        except Exception: z = np.nan
        if DEVICE == "cuda": torch.cuda.empty_cache()
        return z
    if FSD_N:
        rng = np.random.default_rng(0); idx = np.concatenate([rng.choice(M.index[M.label == k], FSD_N // 2, replace=False) for k in (0, 1)])
    else: idx = M.index.values
    for c in CONDS:
        if f"f_{c}" not in M: M[f"f_{c}"] = np.nan
        for i in tqdm(idx, desc=f"FSD {c}"):
            if np.isnan(M.at[i, f"f_{c}"]): M.at[i, f"f_{c}"] = f_score(M.at[i, f"p{c}"])
        M.to_csv(f"{OUT}/resultados_nativo.csv", index=False)
print("pronto")

In [ ]:
# 5) AUC com IC95% por condicao: conjunto todo e subconjunto pareado por escore de propensao (por condicao)
def boot(yy, s, n=2000, seed=0):
    rng = np.random.default_rng(seed); v = []
    for _ in range(n):
        i = rng.integers(0, len(yy), len(yy))
        if len(set(yy[i])) == 2: v.append(roc_auc_score(yy[i], s[i]))
    return np.percentile(v, [2.5, 97.5])
def pareia(Xd, yd, cal=0.05):
    _, ps = audit(Xd, yd); F = pd.DataFrame(dict(ps=ps, y=yd)); reais = F[F.y == 0].sort_values("ps"); usadas = set(); pares = []
    for i, r in F[F.y == 1].sort_values("ps").iterrows():
        cand = reais[~reais.index.isin(usadas)]
        if len(cand) == 0: break
        j = (cand.ps - r.ps).abs().idxmin()
        if abs(cand.loc[j, "ps"] - r.ps) < cal: usadas.add(j); pares += [i, j]
    return pares
rows = []
for c in [k for k in "CAD" if f"u_{k}" in M]:
    pares = pareia(FEAT[c], y); auc_par = audit(FEAT[c][pares], y[pares])[0] if len(pares) > 40 else np.nan
    for nm, col, sg in (("UnivFD", f"u_{c}", 1), ("FSD", f"f_{c}", -1)):
        if col not in M: continue
        for nome, sub in (("todas", M), ("pareado", M.loc[pares])):
            d = sub.dropna(subset=[col])
            if d.label.nunique() < 2 or len(d) < 20: continue
            s = sg * d[col].values; lo, hi = boot(d.label.values, s)
            rows.append(dict(condicao=c, detector=nm, conjunto=nome, n=len(d), AUC=round(roc_auc_score(d.label, s), 3), IC95=f"[{lo:.2f}, {hi:.2f}]", AUC_atalhos_conjunto=round(auc_par, 3) if nome == "pareado" else np.nan))
R = pd.DataFrame(rows); print(R.to_string(index=False)); R.to_csv(f"{OUT}/resumo_nativo.csv", index=False)
print("\nLeitura: condicao C e o controle positivo. Se mesmo ali o AUC ficar perto de 0,5 ou abaixo, os detectores nao enxergam essas geracoes; se subir, a compressao JPEG era o problema.")